# v10 — probe lỗi generate + đo R@5 thật trên pool hybrid

**Hai phép đo trong một lần chạy.** Cả hai cần T4 và cần cùng một bước dựng dữ liệu
(~12 phút CPU); chạy hai notebook riêng là trả phí đó hai lần.

| phần | câu hỏi | chặn việc gì |
|---|---|---|
| **A** | Generator ra lặp từ + chữ Hán khi nhồi >1 Evidence — có phải tràn fp16? | toàn bộ tầng prompt/SFT |
| **B** | Pool hybrid nâng R@5 thật bao nhiêu? (§15.8 mới **ước** +0,9) | không chặn, nhưng là số nợ cuối của §15 |

A chạy TRƯỚC vì nó đang chặn. Session chết giữa đường thì vẫn có câu trả lời A.

Cần gắn trước khi chạy:

| loại | thứ cần gắn |
|---|---|
| Accelerator | **GPU T4 x2** (P100 là sm_60, fp16 vỡ — cell 2 báo) |
| Internet | **ON** (tải reranker từ HuggingFace ở phần B) |
| Model | `sunny070306/qwen3-5-2b-base` |
| Dataset | `official` · `legalqa-v10-src` · `legalqa-warmup` |


## 1 — Nạp codebase từ Dataset

Dòng `codebase:` in commit đang chạy. Lệch với `git log -1` trên máy dev = dataset cũ,
chạy `./scripts/push_dataset.sh` rồi *Add Input* lại bản mới.


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Không clone GitHub nữa: repo private, fine-grained PAT + Kaggle Secrets tốn hai vòng
# debug mà vẫn 404. Codebase đi vào bằng Kaggle Dataset — không auth, chạy được cả khi
# Internet OFF. Đẩy bản mới bằng scripts/push_dataset.sh trên máy dev.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if cand.name == "legalqa" and (cand / "bm25.py").exists():
        SRC = cand.parent.parent          # .../src/legalqa -> gốc repo
        break
if SRC is None:
    # Kaggle đôi khi giữ nguyên .zip thay vì bung; tự bung vào working.
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase trong /kaggle/input — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

# Copy sang working: scripts ghi bm25.pkl/chunks.jsonl, /kaggle/input là read-only.
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

# 4.44.2 (bản ghim ở eval_rerank.ipynb cho XLM-R) KHÔNG biết kiến trúc qwen3_5 ->
# KeyError: 'qwen3_5' ở CONFIG_MAPPING. 5.14.1 là bản `main` đã ghim cho đúng model này
# (requirements.txt, build_wheelhouse.sh) và v9.2 chạy cả XLM-R lẫn Qwen trên nó.
# dense.py/rerank.py không dùng torch_dtype nên không vướng đổi API 4.x -> 5.x.
!pip -q install transformers==5.14.1 tokenizers==0.22.2 sentencepiece

os.chdir(REPO)
sys.path.insert(0, "src")

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip() if (REPO / "COMMIT.txt").exists() else "KHÔNG có COMMIT.txt")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator")
print("số GPU  :", torch.cuda.device_count())

# P100 (sm_60) không chạy được fp16 với torch bản trên Kaggle. Code tự tụt về
# fp32 nên không vỡ, nhưng chậm ~2x — báo to để biết mà đổi Accelerator.
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("sm      :", f"{cap[0]}.{cap[1]}", "(fp16 OK)" if cap[0] >= 7 else "(fp16 KHÔNG chạy)")
    # assert chứ không chỉ cảnh báo, KHÁC hai notebook kia: ở đó fp32-fallback chỉ chậm
    # ~2x nhưng số đo vẫn đúng. Ở đây phần A đo CHÍNH fp16, nên P100 không cho ra số tệ
    # hơn — nó không cho ra số nào cả. Lượt trước phải bấm hủy tay ở phút 1.
    assert cap[0] >= 7, (
        f"sm_{cap[0]}{cap[1]} không chạy được fp16 -> phần A vô nghĩa. "
        "Settings -> Accelerator -> GPU T4 x2, rồi Save & Run All."
    )

# Vỡ ở phút 1 thay vì phút 25: kiểm transformers nhận diện được kiến trúc Qwen TRƯỚC khi
# dựng dữ liệu. Lượt trước chết ở đây sau khi đã đốt 25 phút clean/chunk/gold.
import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers:", transformers.__version__)
assert "qwen3_5" in CONFIG_MAPPING, (
    f"transformers {transformers.__version__} không biết qwen3_5 — phần A sẽ vỡ. "
    "Restart session rồi chạy lại cell này."
)
print("qwen3_5     : CONFIG_MAPPING OK")

# Phần B ĐÃ ĐO XONG (§15.8: R@5 hybrid 93,6% vs BM25 thuần 92,0%, chênh +1,6). Lượt này
# chỉ chạy lại phần A — nó vỡ vì `KeyError: 'qwen3_5'`. Để CHAY_B = True thì tốn thêm
# 12 phút dựng dữ liệu + ~40 phút rerank để in ra đúng những số đã có trong docs.
CHAY_B = False
print("CHAY_B:", CHAY_B)


## 2 — Tìm dữ liệu thi


In [ ]:
if CHAY_B:  # phần A dùng prompt tự sinh trong probe, không cần corpus
    from pathlib import Path

    cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
    assert cands, "Không thấy selected-contexts/ trong /kaggle/input — chưa gắn dữ liệu thi"
    DATA_DIR = cands[0]

    train = DATA_DIR / "train.json"
    assert train.exists(), f"Thiếu train.json cạnh selected-contexts/ trong {DATA_DIR}"

    os.environ["DATA_DIR"] = str(DATA_DIR)
    os.environ["TRAIN_JSON"] = str(train)
    os.environ["CLEAN_DIR"] = "/kaggle/working/clean"
    os.environ["PYTHONPATH"] = "src"

    n = len(list((DATA_DIR / "selected-contexts").glob("*.json")))
    print(f"DATA_DIR  {DATA_DIR}")
    print(f"corpus    {n:,} file  (máy dev: 8.532)")
else:
    print('bỏ qua — CHAY_B=False')


## 3 — clean → chunk → gold (CPU, ~12 phút)

Ba số phải khớp máy dev, lệch là dữ liệu khác bản:
`Document 8.512` · `Chunk 287.358` · `gold 6.250/7.000 = 89,3%`


In [ ]:
if CHAY_B:  # 12 phút dựng dữ liệu chỉ để phục vụ phần B
    !mkdir -p $CLEAN_DIR
    !python scripts/clean_data.py
    !python scripts/build_chunks.py
    !python scripts/build_gold.py
    # warmup.json: 500 câu CÓ Reference answer và KHÔNG nằm trong train. Mọi ngưỡng pool/K
    # đang chốt trên train -> cần tập độc lập kiểm lại, nếu không là chốt tham số trên chính
    # tập đã dùng để quan sát. Máy dev dựng được 456/500 = 91,2%.
    #
    # Nó KHÔNG nằm cạnh train.json (dataset `official` chỉ có official/) -> dò cả /kaggle/input.
    from pathlib import Path
    warm = next(Path("/kaggle/input").glob("**/warmup.json"), None)
    if warm:
        !TRAIN_JSON=$warm GOLD_OUT=gold_warmup.jsonl python scripts/build_gold.py
        os.environ["CO_WARMUP"] = "1"
    else:
        os.environ["CO_WARMUP"] = ""
        print("KHÔNG thấy warmup.json — gắn dataset legalqa-warmup để có tập độc lập")
else:
    print('bỏ qua — CHAY_B=False')


# PHẦN A — generator có tràn fp16 không

Bug: nhồi >1 Evidence thì generator lặp từ và sinh chữ Hán. Không mới — v9.2 đã gặp và
vá triệu chứng: `verifier.py:31` có regex trừ điểm câu chứa CJK `\u4e00-\u9fff`, và
`PIPELINE_V9.md` ghi "Qwen output được xoá câu lặp". Vá hai triệu chứng, không chữa nguyên nhân.

Giả thuyết: T4 là **sm_75** nên `is_bf16_supported()` = `False`, config buộc fp16
(`kaggle_inference_t4.yaml:20`). Activation Qwen vượt trần fp16 **65.504** → `inf` →
`NaN` sau softmax → logits phẳng → decode lặp và trôi về token tần suất cao của
pretraining, với Qwen là tiếng Trung.

Mảnh bằng chứng đã loại được nhánh sampling: `repetition_penalty 1.1` và
`no_repeat_ngram_size 5` **đã bật** mà vẫn lặp. Logits processor cộng/trừ trên số hữu
hạn — nó không sửa được `NaN`.

## A1 — Tìm checkpoint Qwen

Dò theo `architectures` trong `config.json` thay vì gõ tay: Kaggle mount model ở
`/kaggle/input/<slug>/<framework>/<instance>/<version>` và framework trong ref là
`TensorFlow2` (nhãn sai từ lúc upload nhưng ref phải giữ đúng) nên đường dẫn khó đoán.

Loại thư mục chỉ có `adapter_config.json`: đó là LoRA, không phải base.


In [ ]:
from pathlib import Path
import json

QWEN = None
for cfg in Path("/kaggle/input").glob("**/config.json"):
    try:
        d = json.loads(cfg.read_text())
    except Exception:
        continue
    if any("Qwen" in a for a in d.get("architectures", [])):
        # Base thật phải có weight cạnh config; adapter LoRA thì không.
        if any(cfg.parent.glob("*.safetensors")) or any(cfg.parent.glob("*.bin")):
            QWEN = cfg.parent
            break
assert QWEN, "Không thấy base Qwen — gắn Model sunny070306/qwen3-5-2b-base"

d = json.loads((QWEN / "config.json").read_text())
print("checkpoint :", QWEN)
print("arch       :", d.get("architectures"))

# Lượt trước in `None` cho cả dtype lẫn maxpos vì đây là config LỒNG: checkpoint là
# `Qwen3_5ForConditionalGeneration` (đa mô thức) nên mọi thứ của LM nằm trong
# `text_config`. Đọc tầng gốc là đọc sai chỗ, không phải checkpoint thiếu trường.
tc = d.get("text_config") or d
print("tầng đọc   :", "text_config" if "text_config" in d else "gốc (không lồng)")
print("dtype gốc  :", tc.get("torch_dtype") or tc.get("dtype") or d.get("torch_dtype") or d.get("dtype"),
      " <- train bằng gì; bf16 mà chạy fp16 là đúng giả thuyết tràn")
print("maxpos     :", tc.get("max_position_embeddings"))
print("layer      :", tc.get("num_hidden_layers"), "· hidden", tc.get("hidden_size"))
os.environ["QWEN"] = str(QWEN)


## A2 — Đo

Ba đại lượng, cùng prompt, hai dtype, ba độ dài (1/3/5 Evidence):

1. `|act| max` so với **trần fp16 = 65.504**
2. số tensor có phần tử không hữu hạn + **tên layer đầu tiên vỡ**
3. 40 token sinh ra + số ký tự CJK

**Đọc kết quả:**

| thấy gì | kết luận |
|---|---|
| fp16 chạm trần, có tensor không hữu hạn, CJK > 0; fp32 sạch | **giả thuyết đúng** — sửa theo `layer_dau` |
| fp32 cũng ra CJK | giả thuyết **sai** — không phải lỗi số; quay lại nghi SFT train trên prompt 1 chunk |
| fp16 sạch ở cả 5 chunk | chưa tái hiện — prompt tổng hợp khác prompt thật, phải thay bằng prompt thật |

`layer_dau` chọn cách sửa: tập trung ở attention → softmax fp32 (~0 chi phí); rải khắp →
NF4 với `bnb_4bit_compute_dtype=float32` (weight 1,3 GB, nhanh hơn fp32 vì nghẽn băng
thông); fallback → generator fp32 (8 GB, vừa T4 16 GB sau khi unload reranker).

Cả ba **không đổi số params**, nên trần 4B không bị ảnh hưởng — quantization đổi
bit/param, không đổi param.


In [ ]:
!PYTHONPATH=src python scripts/probe_fp16_overflow.py --model $QWEN --n-chunk 1 3 5


# PHẦN B — R@5 thật trên pool hybrid

§15.8 đo **trần** pool: 96,7% → 97,9% (+1,2 điểm), rồi suy R@5 qua hệ số 0,72 của §15.6
→ ước +0,9. Đó là **ước, không phải đo**: hệ số 0,72 đo trên pool BM25 thuần, không có
gì bảo đảm reranker ăn cùng tỉ lệ khoảng trống trên pool đã trộn.

B1 encode 287k chunk (e5-small, `max_len 256` — §15.8 chốt 256 vì 512 tệ hơn VÀ chậm
hơn gấp đôi, **đừng thử lại 512**). ~7 phút T4.


In [ ]:
if CHAY_B:  # phần B
    !PYTHONPATH=src python scripts/build_dense.py --max-len 256
else:
    print('bỏ qua — CHAY_B=False')


## B2 — cross-encoder trên pool hybrid

Mốc phải đấu (pool BM25 thuần, §15.3): **R@1 73,1% · R@5 92,2% · trần 96,7%**

Dòng `trần` phải ra **~97,9%**. Nếu nó vẫn 96,7% thì nhánh dense không vào được pool và
mọi số sau vô nghĩa — kiểm `dense.npy`/`dense.ids` có sinh ra chưa.


In [ ]:
if CHAY_B:  # phần B
    from huggingface_hub import snapshot_download

    RR = snapshot_download("AITeamVN/Vietnamese_Reranker")
    os.environ["RR"] = RR
    print(RR)
else:
    print('bỏ qua — CHAY_B=False')


In [ ]:
if CHAY_B:  # phần B
    !PYTHONPATH=src python scripts/eval_rerank.py --model $RR --pool 50 --n-cau 500 \
        --batch-size 32 --dense dense.npy
else:
    print('bỏ qua — CHAY_B=False')


## B3 — cùng số câu, pool BM25 thuần, để trừ ra

Cùng `n-cau`, cùng model, khác pool. Chênh R@5 giữa B2 và B3 **chính là** con số đang nợ.

Không lấy số §15.3 cũ ra trừ: lượt đó khác lượt này ở seed và ở máy, nên chênh do nhiễu
sẽ lẫn vào chênh do dense. Đo cặp trong cùng một lần chạy thì phần nhiễu chung triệt tiêu.


In [ ]:
if CHAY_B:  # phần B
    !PYTHONPATH=src python scripts/eval_rerank.py --model $RR --pool 50 --n-cau 500 --batch-size 32
else:
    print('bỏ qua — CHAY_B=False')
